# AI & ML Internship — Task 1
## Data Cleaning & Preprocessing

**Objective:** Clean and prepare raw data for ML using the Titanic dataset.

### Steps covered
1. Import and explore data
2. Handle missing values
3. Encode categorical features
4. Standardize numerical features
5. Visualize and remove outliers

In [ ]:
from pathlib import Path
import urllib.request
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
DATA_PATH = ROOT / "data" / "titanic.csv"
DATA_PATH.parent.mkdir(exist_ok=True)
URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"

if not DATA_PATH.exists():
    urllib.request.urlretrieve(URL, DATA_PATH)

df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
print("Shape:", df.shape)
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())

## 1. Missing-value handling
`deck` has substantial missing information, so it is removed. Numerical missing values are filled with the median and categorical missing values with the mode.

In [ ]:
df = df.drop(columns=["deck"], errors="ignore")

for col in ["age", "fare"]:
    df[col] = df[col].fillna(df[col].median())

for col in ["embarked", "embark_town"]:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].mode()[0])

print(df.isnull().sum())

## 2. Categorical encoding
The target is `survived`. The `alive` column duplicates the target, so it is excluded. Other categorical columns are one-hot encoded.

In [ ]:
target = df["survived"].copy()
features = df.drop(columns=["survived", "alive"], errors="ignore")

categorical_cols = features.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
numerical_cols = features.select_dtypes(include=np.number).columns.tolist()

encoded = pd.get_dummies(features, columns=categorical_cols, drop_first=True, dtype=int)
print("Categorical columns:", categorical_cols)
print("Numerical columns:", numerical_cols)
print("Encoded shape:", encoded.shape)

## 3. Outlier detection with boxplots
The IQR rule uses values below Q1 − 1.5×IQR or above Q3 + 1.5×IQR as outliers.

In [ ]:
outlier_mask = pd.Series(False, index=encoded.index)
outlier_counts = {}

for col in numerical_cols:
    q1 = encoded[col].quantile(0.25)
    q3 = encoded[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    col_outliers = (encoded[col] < lower) | (encoded[col] > upper)
    outlier_counts[col] = int(col_outliers.sum())
    outlier_mask |= col_outliers

    plt.figure(figsize=(7, 4))
    sns.boxplot(x=encoded[col])
    plt.title(f"Boxplot - {col}")
    plt.show()

print("Outlier counts:", outlier_counts)
print("Rows containing at least one outlier:", int(outlier_mask.sum()))

## 4. Remove outliers and standardize numerical features

In [ ]:
cleaned = encoded.loc[~outlier_mask].copy()
cleaned_target = target.loc[cleaned.index]

scale_cols = [c for c in numerical_cols if c in cleaned.columns]
scaler = StandardScaler()
cleaned[scale_cols] = scaler.fit_transform(cleaned[scale_cols])

cleaned["survived"] = cleaned_target.astype(int)

print("Original shape:", df.shape)
print("Final shape:", cleaned.shape)
print("Missing values remaining:", int(cleaned.isnull().sum().sum()))

output_path = ROOT / "outputs" / "titanic_preprocessed.csv"
output_path.parent.mkdir(exist_ok=True)
cleaned.to_csv(output_path, index=False)
print("Saved to:", output_path)

## Interview Questions — Short Answers

1. **Types of missing data:** MCAR (Missing Completely At Random), MAR (Missing At Random), and MNAR (Missing Not At Random).
2. **Categorical variables:** Use one-hot encoding for nominal categories; label/ordinal encoding can be used when categories have meaningful order.
3. **Normalization vs standardization:** Normalization commonly rescales values to a fixed range such as 0–1. Standardization transforms values to approximately mean 0 and standard deviation 1.
4. **Outlier detection:** IQR/boxplots, z-scores, and domain rules are common methods.
5. **Why preprocessing matters:** It improves data quality and makes features suitable for many ML algorithms.
6. **One-hot vs label encoding:** One-hot creates separate binary columns; label encoding maps categories to integer codes.
7. **Data imbalance:** Resampling, class weights, suitable metrics, and threshold adjustment are common approaches.
8. **Can preprocessing affect accuracy?** Yes. Poor preprocessing can introduce leakage, distort features, or reduce useful information; appropriate preprocessing can improve model performance and reliability.